In [1]:
import pandas as pd

In [ ]:
import os
# os.chdir("../")

In [3]:
df = pd.read_csv(r"artifacts\data_ingestion\stock_data.csv")
df.head()

,Date,Close,High,Low,Open,Volume
0,2000-01-03,0.837724,0.841934,0.761015,0.784870,535796800
1,2000-01-04,0.767096,0.827901,0.757273,0.810127,512377600
2,2000-01-05,0.778321,0.827434,0.770838,0.776450,778321600
3,2000-01-06,0.710966,0.800773,0.710966,0.794225,767972800
4,2000-01-07,0.744644,0.755870,0.714709,0.722192,460734400


In [4]:
list(df.columns)

['Date', 'Close', 'High', 'Low', 'Open', 'Volume']

In [5]:
df

,Date,Close,High,Low,Open,Volume
0,2000-01-03,0.837724,0.841934,0.761015,0.784870,535796800
1,2000-01-04,0.767096,0.827901,0.757273,0.810127,512377600
2,2000-01-05,0.778321,0.827434,0.770838,0.776450,778321600
3,2000-01-06,0.710966,0.800773,0.710966,0.794225,767972800
4,2000-01-07,0.744644,0.755870,0.714709,0.722192,460734400
...,...,...,...,...,...,...
6680,2026-07-28,340.079987,342.890015,335.600006,340.029999,51859000
6681,2026-07-29,338.190002,344.570007,337.350006,339.730011,56090800
6682,2026-07-30,333.429993,334.750000,329.589996,333.100006,74817800
6683,2026-07-31,308.910004,310.690002,300.000000,304.809998,132489100


In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    raw_data_file: Path
    status_file: Path
    ge_expectation_suite: str
    all_schema: dict

In [9]:
from stock_prediction.utils.common import *
from stock_prediction.constants import *

class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.schema = read_yaml(schema_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
    def get_data_validation_config(self):
        config = self.config.data_validation
        schema = self.schema.columns
        create_directories([config.root_dir])
        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            raw_data_file=Path(config.raw_data_file),
            status_file=Path(config.status_file),
            ge_expectation_suite=config.ge_expectation_suite,
            all_schema=schema
        )
        return data_validation_config
    

In [11]:
df = pd.read_csv(r"artifacts\data_ingestion\stock_data.csv", parse_dates=["Date"])
df.head()

,Date,Close,High,Low,Open,Volume
0,2000-01-03,0.837724,0.841934,0.761015,0.784870,535796800
1,2000-01-04,0.767096,0.827901,0.757273,0.810127,512377600
2,2000-01-05,0.778321,0.827434,0.770838,0.776450,778321600
3,2000-01-06,0.710966,0.800773,0.710966,0.794225,767972800
4,2000-01-07,0.744644,0.755870,0.714709,0.722192,460734400


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6685 entries, 0 to 6684
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    6685 non-null   datetime64[ns]
 1   Close   6685 non-null   float64       
 2   High    6685 non-null   float64       
 3   Low     6685 non-null   float64       
 4   Open    6685 non-null   float64       
 5   Volume  6685 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 313.5 KB


In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
df.Date.nunique()

6685

In [15]:
df.isna().sum()

Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [16]:
df.iloc[:, 1:].min()

Close     1.963766e-01
High      1.974243e-01
Low       1.903894e-01
Open      1.944304e-01
Volume    1.791060e+07
dtype: float64

In [18]:
import great_expectations as gx
context = gx.get_context()
suite = context.suites.add(
    gx.ExpectationSuite(name="stock_data_suite")
)


data_source = context.data_sources.add_pandas_filesystem(
    name="stock_data_source",
    base_directory=r"artifacts\data_ingestion"
)
data_asset = data_source.add_csv_asset(
    name="stock_data.csv",

)
batch_definition = data_asset.add_batch_definition_path(
    name="stock_data_batch",
    path="stock_data.csv"
)

2026-08-09 00:10:10,215 | INFO | context_factory| Could not find local file-backed GX project
2026-08-09 00:10:10,218 | INFO | base| Created temporary directory 'C:\Users\Ibk\AppData\Local\Temp\tmp51tg62e4' for ephemeral docs site
2026-08-09 00:10:10,221 | INFO | config| Loading 'datasources' ->
[]
2026-08-09 00:10:10,241 | INFO | sources| 'stock_data_source' PandasFilesystemDatasource uses FilesystemDataConnector


In [178]:
# for result in validation_results.results:
#     print(
#         result.expectation_config.type,
#         result.success,
#         result.result
#     )

In [208]:
columns = list(df.columns)

suite.add_expectation(
    gx.expectations.ExpectTableColumnsToMatchSet(
        column_set=columns,
        exact_match=True
    )
)
for column in columns:
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(
            column=column
        )
)
suite

{
  "name": "stock_data_suite",
  "id": "14229668-581f-4cf4-84c4-dfa555e54def",
  "expectations": [
    {
      "type": "expect_table_columns_to_match_set",
      "kwargs": {
        "column_set": [
          "Date",
          "Close",
          "High",
          "Low",
          "Open",
          "Volume"
        ]
      },
      "meta": {},
      "id": "45da17da-ef8a-4a61-9377-93ed3ac014c5",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "Date"
      },
      "meta": {},
      "id": "2594fd7f-6f8b-4e6f-93fe-33c97a3248c1",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "Close"
      },
      "meta": {},
      "id": "f55df4cd-abfa-4214-a7ba-43853c48bbd2",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "High"
      },
      "meta": {},
   

In [209]:
for column in columns:
    if column != "Date":
        suite.add_expectation(
            gx.expectations.ExpectColumnValueLengthsToBeBetween(
                column=column,
                min_value=0
                )
            )
        


In [210]:
context.suites.add_or_update(suite)

{
  "name": "stock_data_suite",
  "id": "14229668-581f-4cf4-84c4-dfa555e54def",
  "expectations": [
    {
      "type": "expect_table_columns_to_match_set",
      "kwargs": {
        "column_set": [
          "Date",
          "Close",
          "High",
          "Low",
          "Open",
          "Volume"
        ]
      },
      "meta": {},
      "id": "45da17da-ef8a-4a61-9377-93ed3ac014c5",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "Date"
      },
      "meta": {},
      "id": "2594fd7f-6f8b-4e6f-93fe-33c97a3248c1",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "Close"
      },
      "meta": {},
      "id": "f55df4cd-abfa-4214-a7ba-43853c48bbd2",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "High"
      },
      "meta": {},
   

In [211]:
validation_definition = gx.ValidationDefinition(
    data=batch_definition,
    suite=suite,
    name="stock_data_validation"
)

In [212]:
context.validation_definitions.add(validation_definition)

ValidationDefinition(name='stock_data_validation', data=BatchDefinition(id=UUID('f08f88f3-9381-48bd-95e9-c1bce47a7ee3'), name='stock_data_batch', partitioner=FileNamePartitionerPath(regex=re.compile('stock_data.csv$'), param_names=(), sort_ascending=True)), suite={
  "name": "stock_data_suite",
  "id": "14229668-581f-4cf4-84c4-dfa555e54def",
  "expectations": [
    {
      "type": "expect_table_columns_to_match_set",
      "kwargs": {
        "column_set": [
          "Date",
          "Close",
          "High",
          "Low",
          "Open",
          "Volume"
        ]
      },
      "meta": {},
      "id": "45da17da-ef8a-4a61-9377-93ed3ac014c5",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "Date"
      },
      "meta": {},
      "id": "2594fd7f-6f8b-4e6f-93fe-33c97a3248c1",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "

In [213]:
valiation_results = validation_definition.run()
valiation_results.success

2026-08-08 23:14:15,883 | INFO | batch_filter| batch_slice: None was parsed to: slice(0, None, None)
2026-08-08 23:14:15,885 | INFO | batch_filter| batch_slice: slice(0, None, None) was parsed to: slice(0, None, None)


Calculating Metrics: 100%|██████████| 63/63 [00:00<00:00, 748.26it/s]


True

In [230]:
import great_expectations as gx
context = gx.get_context()



# data_source = context.data_sources.add_pandas_filesystem(
#     name="stock_data_source",
#     base_directory=r"artifacts\data_ingestion"
# )
# data_asset = data_source.add_csv_asset(
#     name="stock_data",

# )
# batch_definition = data_asset.add_batch_definition_path(
#     name="stock_data_batch",
#     path="stock_data.csv"
# )

data = pd.read_csv("artifacts/data_ingestion/stock_data.csv", parse_dates=["Date"])
columns = list(data.columns)

data_source = context.data_sources.add_pandas("stock_data_source")

data_asset = data_source.add_dataframe_asset(name="stock_data_asset")
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    "stock_data_batch"
)
batch = batch_definition.get_batch(batch_parameters={"dataframe": data})

suite = context.suites.add(
    gx.ExpectationSuite(name="stock_data_suite")
)
# batch_request = data_asset.build_batch_request()
# batch = data_asset.get_batch(batch_request)


suite.add_expectation(
    gx.expectations.ExpectTableColumnsToMatchSet(
        column_set=columns,
        exact_match=True
    )
)

for column in columns:
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(
            column=column
        )
)


for column in columns:
    if column != "Date":
        suite.add_expectation(
            gx.expectations.ExpectColumnValuesToBeBetween(
                column=column,
                min_value=0
                )
            )
        
context.suites.add_or_update(suite)


results = batch.validate(suite)

# context.validation_definitions.add(validation_definition)

# validation_results = validation_definition.run()
results.success
if results.success:
    print("success")
else:
    print("failed")


2026-08-08 23:49:13,644 | INFO | context_factory| Could not find local file-backed GX project
2026-08-08 23:49:13,654 | INFO | base| Created temporary directory 'C:\Users\Ibk\AppData\Local\Temp\tmpe7clurbd' for ephemeral docs site
2026-08-08 23:49:13,657 | INFO | config| Loading 'datasources' ->
[]


Calculating Metrics: 100%|██████████| 58/58 [00:00<00:00, 600.07it/s]

success


In [226]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6685 entries, 0 to 6684
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    6685 non-null   datetime64[ns]
 1   Close   6685 non-null   float64       
 2   High    6685 non-null   float64       
 3   Low     6685 non-null   float64       
 4   Open    6685 non-null   float64       
 5   Volume  6685 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 313.5 KB


In [ ]:

data = pd.read_csv("artifacts/data_ingestion/stock_data.csv", parse_dates=["Date"])
columns = list(data.columns)

data_source = context.data_sources.add_pandas("stock_data_source")

data_asset = data_source.add_dataframe_asset(name="stock_data_asset")
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    "stock_data_batch"
)
batch = batch_definition.get_batch(batch_parameters={"dataframe": data})


In [45]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config
    
    def validate_data(self):
        try:
            data = pd.read_csv(self.config.raw_data_file, parse_dates=["Date"])
            columns = list(data.columns)
            context = gx.get_context()
            data_source = context.data_sources.add_pandas("stock_data_source")
            data_asset = data_source.add_dataframe_asset(name="stock_data_asset")
            batch_definition = data_asset.add_batch_definition_whole_dataframe("stock_data_batch")
            batch = batch_definition.get_batch(batch_parameters={"dataframe": data})
            suite = context.suites.add(
                gx.ExpectationSuite(self.config.ge_expectation_suite)
            )

            suite.add_expectation(
                gx.expectations.ExpectTableColumnsToMatchSet(
                    column_set=columns,
                    exact_match=True
                )
            )

            for column in columns:
                suite.add_expectation(
                    gx.expectations.ExpectColumnValuesToNotBeNull(
                        column=column
                    )
            )


            for column in columns:
                if column != "Date":
                    suite.add_expectation(
                        gx.expectations.ExpectColumnValuesToBeBetween(
                            column=column,
                            min_value=0
                            )
                        )
                    
            context.suites.add_or_update(suite)

            results = batch.validate(suite)
            validation_status = bool(results.success)
            self._write_status(validation_status)
            if validation_status:
                logger.info(f"Validation passed")
            else:
                logger.error(f"Data validation failed: {results}")
            return validation_status
        except Exception as e:
            logger.error(f"Error encountered during validation: {e}")
            raise e
    def _write_status(self, status: bool):
        with open(self.config.status_file, "w") as f:
            f.write(str(status))

In [46]:
config = ConfigurationManager()
data_validation_config = config.get_data_validation_config()
data_validation = DataValidation(data_validation_config)
data_validation.validate_data()

2026-08-09 00:30:27,469 | INFO | common| YAML file: config\config.yaml loaded successfully.
2026-08-09 00:30:27,474 | INFO | common| YAML file: schema.yaml loaded successfully.
2026-08-09 00:30:27,477 | INFO | common| YAML file: params.yaml loaded successfully.
2026-08-09 00:30:27,479 | INFO | common| Directory created at: artifacts
2026-08-09 00:30:27,481 | INFO | common| Directory created at: artifacts/data_validation
2026-08-09 00:30:27,512 | INFO | context_factory| Could not find local file-backed GX project
2026-08-09 00:30:27,514 | INFO | base| Created temporary directory 'C:\Users\Ibk\AppData\Local\Temp\tmphfuj7db5' for ephemeral docs site
2026-08-09 00:30:27,517 | INFO | config| Loading 'datasources' ->
[]


Calculating Metrics: 100%|██████████| 58/58 [00:00<00:00, 1192.94it/s]

2026-08-09 00:30:27,780 | INFO | 3571773912| Validation passed


True

SyntaxError: invalid syntax (3035376500.py, line 1)